In [1]:
import sys; sys.path.append('..')
from osp import *

In [2]:
for k,v in STASH_FEAT_EXAMPLES2.items(): break
k, v[0]

('phil/10.2307/20111773__02',
 {'feature': 'deprel_det',
  'word': 'an',
  'count': 10,
  'perc': 8.7,
  'eg_text': 'To make this plain by *AN* example, suppose a geometri cian is demonstrating the m',
  'eg_html': '<span data-word-id="1" data-word-pos="TO" data-word-deprel="mark" data-word-color-by="weight_z" data-word-color-val="3.473" style="background-color: #053061; color: white; font-size: 1.2em; font-weight: normal; display: inline-block; text-align: left; vertical-align: top; line-height: 1.2; padding: 2px 4px; border-radius: 4px; margin: 2px;">To<sub style="display: block; font-size: 0.6em; opacity: 0.7; line-height: 1; font-weight: normal; padding: 2px; font-family: monospace;">TO/mark</sub></span><span data-word-id="2" data-word-pos="VB" data-word-deprel="advcl" data-word-color-by="weight_z" data-word-color-val="0.071" style="background-color: #f0f4f6; color: black; font-size: 1.2em; font-weight: normal; display: inline-block; text-align: left; vertical-align: top; line-heig

In [3]:
# get_current_feat_weights().index.tolist()

In [4]:
# FEAT2DESC

In [5]:
lines="""That Arthur *INHERITED* at least one of the talismans of the lords of fa
Like Cuchulinn and Cormac he gradually *INHERITED* the treasures and the talismans of older lords o
s a proof that bleeding lance and grail *WERE* in origin fairy objects. 
This *WAS* neither intended nor generally taken seriously, though
As Gaston Paris *SAID*, "Ce moyen age ... traduisait milites par chevaliers 
Paganism *WAS* dead, to be sure, but its professors in the Troilus we
Again, the lost scene in the W stanza *WAS* in itself, probably, a revision of a still older scene
It is hardly possible that he *ALTERED* the position of the foot-washing episode. 
Probably the first *WAS* the case. 
r later called the "Faerie Queene," she *HAD* "gossip'd"I with her and sat with her "on Neptune's ye""".split('\n')

In [31]:
def center_starred_keyword(line, window=50, keep_asterisks=False):
    radius = window // 2
    m = re.search(r"\*[A-Z][A-Z0-9_-]*\*", line)
    if not m:
        return line
    start, end = m.span()
    left_text = line[start-radius if start-radius > 0 else 0:start]
    right_text = line[end:end+radius]
    word_text = line[start:end]
    word_text_unstarred = word_text[1:-1]
    
    if len(left_text) < radius:
        needs_padding = radius - len(left_text)
        left_text = ' '*needs_padding + left_text

    word = word_text_unstarred if not keep_asterisks else word_text
    return left_text + word + right_text

In [32]:
for x in lines:
    print(center_starred_keyword(x,50))

             That Arthur INHERITED at least one of the tali
 and Cormac he gradually INHERITED the treasures and the ta
bleeding lance and grail WERE in origin fairy objects.
                    This WAS neither intended nor gen
         As Gaston Paris SAID, "Ce moyen age ... tradu
                Paganism WAS dead, to be sure, but it
st scene in the W stanza WAS in itself, probably, a r
 hardly possible that he ALTERED the position of the foot
      Probably the first WAS the case. 
the "Faerie Queene," she HAD "gossip'd"I with her and


In [70]:
pad_starred_keyword(lines[0],40)

'                            That Arthur INHERITED at least one of the talismans of the lords of fa'

In [54]:
lines[0][12:23]

'*INHERITED*'

In [32]:
def get_slice_feat_egs(slice_ids=None, feats=None, num_egs=10, max_slices=1000):
    # from .slices import get_slice_ids
    if slice_ids is None:
        slice_ids = get_parsed_slice_ids()
    if isinstance(feats, str):
        feats = [feats]
    elif not feats:
        feats = [x for x in FEAT2DESC.keys() if x.split('_')[0] in {'pos','deprel'} and x not in BAD_SLICE_FEATS]
    
    random.shuffle(slice_ids)
    egs = {feat:[] for feat in feats}
    iterr = slice_ids[:max_slices]
    # iterr = tqdm(slice_ids[:max_slices])
    for slice_id in iterr:
        min_feat_len = min(len(egs[f]) for f in egs)
        # min_feat_names = [f for f in egs if len(egs[f]) == min_feat_len]
        # min_feat_name = ', '.join(min_feat_names)
        
        # iterr.set_description(f'min = {min_feat_len} ({min_feat_name[:100]})')
        if min_feat_len >= num_egs:
            break
        
        res_ld = STASH_FEAT_EXAMPLES2.get(slice_id, None)
        if isinstance(res_ld, list) and res_ld:
            for d in res_ld:
                feat = d['feature']
                if (not feats or feat in set(feats)) and len(egs[feat]) < num_egs:
                    d['slice_id'] = slice_id
                    egs[feat].append(d)
    return pd.DataFrame([vx for vl in egs.values() for vx in vl])

In [47]:
df=get_slice_feat_egs(max_slices=1000, num_egs=100)

In [48]:
for g,gdf in df.groupby('feature'): break
htmls = '<br/><br/>'.join([x for x in gdf.eg_html if x])
HTML(htmls)